In [41]:
# Do this once to prepare env
!module load python gcc opencv/4.12.0
!pip install vllm torch transformers accelerate safetensors ipykernel sentencepiece tiktoken blobfile bitsandbytes

---

In [1]:
import os
import json
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM

is_cuda = torch.cuda.is_available()
num = torch.cuda.device_count()
name = torch.cuda.get_device_name(0) if is_cuda and num > 0 else "None"

print(f"CUDA available: {is_cuda}")
print(f"GPU count: {num}")
print(f"Primary GPU: {name}")

device = "cuda"


hub = Path(os.path.expanduser("~/.cache/huggingface/hub"))
base = hub / "models--codellama--CodeLlama-7b-Instruct-hf"

# Prefer the ref in refs/main; fall back to the newest snapshot
ref_file = base / "refs" / "main"
commit = ref_file.read_text().strip()

MODEL_PATH = str(base / "snapshots" / commit)
print("MODEL_PATH =", MODEL_PATH)

# tokenizer
tok = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    use_fast=True,
    local_files_only=True,
)

CUDA available: True
GPU count: 1
Primary GPU: NVIDIA H100 80GB HBM3 MIG 3g.40gb
MODEL_PATH = /home/rpinter/.cache/huggingface/hub/models--codellama--CodeLlama-7b-Instruct-hf/snapshots/22cb240e0292b0b5ab4c17ccd97aa3a2f799cbed


---

Can we make it even faster by compiling the model and using flash attention?


https://arxiv.org/pdf/2205.14135 *We propose FlashAttention, an IO-aware exact attention algorithm that uses tiling to reduce the number of memory reads/writes between GPU high bandwidth memory (HBM) and GPU on-chip SRAM*

In [2]:
from transformers import BitsAndBytesConfig

q_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_use_double_quant=True,
   bnb_4bit_compute_dtype=torch.bfloat16
) 

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    device_map="auto",
    low_cpu_mem_usage=True,
    quantization_config=q_config,
    attn_implementation="sdpa", # flash attention
    dtype=torch.bfloat16, # and changing dtype to lower precision
)

def ask_q(prompt, max_new_tokens=128):
    inst = f"<s>[INST] {prompt.strip()} [/INST]"
    inputs = tok(inst, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=tok.eos_token_id,
            use_cache=True,
        )
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [43]:
def read_file_to_string(path):
    with open(path) as f:
        return f.read()

def get_json_template(n):
    return json.dumps(
        {f'{k}': "<true/false>" for k in range(1,n+1)},
        indent=2,
        ensure_ascii=False)

In [4]:
code_str = read_file_to_string("../examples/plein014-core.js")

In [63]:
material_process = [
    "1) Does it use external audio file? [True/False]",
    "2) Does it use external image file? [True/False]",
    "3) Does it generate sound? [True/False]",
    "4) Does it generate images? [True/False]",
    "5) Does it contain randomness? [True/False]",
    "6) Does it contain interactions? [True/False]",
]
env_interactions = [
    "1) Does it depend on human interaction (through mouse, midi controller, microphone, keyboard, camera, motion sensor, or lidar) to run? [True/False]",
    "2) Does it depend on computer interaction (specifically through data file, data stream, remote data, or web API) to run? [True/False]",
]

env_interactions_2 = [
    "1) Does it depend on human interaction (through mouse, midi controller, microphone, keyboard, camera, motion sensor, or lidar) to run? [True/False]",
    "2) Does it depend on computer interaction through data files to run? [True/False]",
    "3) Does it depend on computer interaction through data stream to run? [True/False]",
    "4) Does it depend on computer interaction through remote data to run? [True/False]",
    "5) Does it depend on computer interaction through web API to run? [True/False]"
]

sensory_outcomes = [
    "1) Does it produce visual sensory outcomes? [True/False]",
    "2) Does it produce auditory sensory outcomes? [True/False]",
    "3) Does it produce physical sensory outcomes? [True/False]",
    "4) Does it produce static or time-based (one of these values is mandatory for this characteristic) sensory outcomes? [True/False]",
]

In [55]:
q_list = material_process
prompt = """
You are a helpful assistent that only answers in valid json files.
Considering this code:
```
{code_str}
```
Create a JSON list with boolean values with the answers for the following questions:

{question}

Now write the valid enumerated json list file with answers and nothing else using the following template:

{json_template}
""".format(
    code_str=code_str, 
    question=q_list, 
    json_template=get_json_template(len(q_list)))

response = ask_q(prompt)

response = json.loads(response)
for q,a in zip(q_list, response.values()):
    print(f"Q: {q} | A: {a}")

Q: 1) Does it use external audio file? [True/False] | A: False
Q: 2) Does it use external image file? [True/False] | A: False
Q: 3) Does it generate sound? [True/False] | A: False
Q: 4) Does it generate images? [True/False] | A: True
Q: 5) Does it contain randomness? [True/False] | A: True
Q: 6) Does it contain interactions? [True/False] | A: False


In [64]:
q_list = env_interactions_2
prompt = """
You are a helpful assistent that only answers in valid json files.
Considering this code:
```
{code_str}
```
Create a JSON list with boolean values with the answers for the following questions:

{question}

Now write the valid enumerated json list file with answers and nothing else using the following template:

{json_template}
""".format(
    code_str=code_str, 
    question=q_list, 
    json_template=get_json_template(len(q_list)))

response = ask_q(prompt)

response = json.loads(response)
for q,a in zip(q_list, response.values()):
    print(f"Q: {q} | A: {a}")

Q: 1) Does it depend on human interaction (through mouse, midi controller, microphone, keyboard, camera, motion sensor, or lidar) to run? [True/False] | A: False
Q: 2) Does it depend on computer interaction through data files to run? [True/False] | A: False
Q: 3) Does it depend on computer interaction through data stream to run? [True/False] | A: False
Q: 4) Does it depend on computer interaction through remote data to run? [True/False] | A: False
Q: 5) Does it depend on computer interaction through web API to run? [True/False] | A: False


In [65]:
q_list = sensory_outcomes
prompt = """
You are a helpful assistent that only answers in valid json files.
Considering this code:
```
{code_str}
```
Create a JSON list with boolean values with the answers for the following questions:

{question}

Now write the valid enumerated json list file with answers and nothing else using the following template:

{json_template}
""".format(
    code_str=code_str, 
    question=q_list, 
    json_template=get_json_template(len(q_list)))

response = ask_q(prompt)

response = json.loads(response)
for q,a in zip(q_list, response.values()):
    print(f"Q: {q} | A: {a}")

Q: 1) Does it produce visual sensory outcomes? [True/False] | A: True
Q: 2) Does it produce auditory sensory outcomes? [True/False] | A: False
Q: 3) Does it produce physical sensory outcomes? [True/False] | A: False
Q: 4) Does it produce static or time-based (one of these values is mandatory for this characteristic) sensory outcomes? [True/False] | A: True
